# Seal or Sea Lion? — train on Colab

Fine-tunes the pinniped classifier on a free Colab GPU. Runtime → Change runtime type → **T4 GPU** (or any GPU).

Steps: clone repo → install deps → re-download the exact dataset from the committed manifest → train → evaluate → export ONNX → parity test → zip the artefacts.

Everything below is the same `scripts/*.py` that run locally on CPU; the notebook is only orchestration.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
%cd /content
!rm -rf pinniped-classifier
!git clone https://github.com/Nalomun/pinniped-classifier.git
%cd pinniped-classifier

Colab ships a CUDA build of torch/torchvision; we keep that and install everything else at the pinned versions.

In [ ]:
!grep -vE '^(torch|torchvision)==' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt
import timm, onnxruntime; print("timm", timm.__version__, "onnxruntime", onnxruntime.__version__)

## Data
`data/manifest.csv` is committed with every photo's URL, license, attribution and split. `--from-manifest` re-downloads exactly those photos (~8k images, ~450 MB, a few minutes). Nothing is re-sampled, so the split is identical to the one in the README.

If you want a *fresh* sample instead, run `python scripts/download.py` (no flag) then `python scripts/split.py`.

In [ ]:
!python scripts/download.py --from-manifest --workers 16

## Train
Defaults: `mobilenetv3_large_100` (ImageNet-21k pretrained), 12 epochs, AdamW + cosine, label smoothing, mixed precision. A few minutes on a T4.

The best checkpoint by validation pinniped accuracy is written to `outputs/best.pt`.

In [ ]:
!python scripts/train.py --model mobilenetv3_large_100 --epochs 12 --batch-size 64 --workers 4

## Evaluate
Writes `reports/metrics.json`, the confusion matrix, worst-misclassification grid, per-species table and the threshold sweep (threshold tuned on val).

In [ ]:
!python scripts/eval.py --workers 4
from IPython.display import Image, display
display(Image("reports/confusion_matrix.png"))
display(Image("reports/worst_misclassifications.png"))

## Export + parity
ONNX (opset 17) + int8 static quantization (kept only if the accuracy drop is < 0.5 points), `model_meta.json`, then the PyTorch ↔ ONNX Runtime parity check.

In [ ]:
!python scripts/export.py
!python scripts/parity.py
!cat export/model_meta.json | head -60

## Grab the artefacts
Download `pinniped-artifacts.zip` (checkpoint, ONNX models, meta, reports) and commit `export/` and `reports/` to the repo.

In [ ]:
!zip -qr pinniped-artifacts.zip outputs/best.pt outputs/history.json export reports
from google.colab import files
files.download("pinniped-artifacts.zip")